# Retrievers (RAG)

From the [Retrievers documentation](../../../docs/source/workflows/retrievers.md):

> *"Retrievers are used to retrieve relevant documents from a vector database."*

## Supported Retriever Providers

| Provider | Type | Description |
|----------|------|-------------|
| **NVIDIA NIM** | `nemo_retriever` | NVIDIA Inference Microservice (NIM) |
| **Milvus** | `milvus_retriever` | Open-source vector database |

## Supported Embedder Providers

From the [Embedders documentation](../../../docs/source/workflows/embedders.md):

| Provider | Type | Description |
|----------|------|-------------|
| **NVIDIA NIM** | `nim` | NVIDIA Inference Microservice (NIM) |
| **OpenAI** | `openai` | OpenAI embedding API |
| **Azure OpenAI** | `azure_openai` | Azure OpenAI embedding API |

## What You'll Learn

1. What is RAG?
2. Milvus retriever configuration
3. Embedder configuration
4. Using retrievers as functions for agents

For more details, see the [Retrievers documentation](../../../docs/source/workflows/retrievers.md) and [Embedders documentation](../../../docs/source/workflows/embedders.md).


In [ ]:
import sys
from pathlib import Path

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


## 1. Embedder Configuration

Embedders convert text to vectors for similarity search:


In [ ]:
from nat.embedder.nim_embedder import NIMEmbedder

# NIM Embedder for NVIDIA embedding models
embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",  # NVIDIA embedding model
    truncate="END",                         # How to handle long text
    name="nim_embedder",
)

print(f"✅ Embedder: {embedder.model_name}")


## 2. Milvus Retriever

Milvus is a popular vector database for RAG:


In [ ]:
from nat.retriever.milvus_retriever import MilvusRetriever

# Milvus retriever configuration
retriever = MilvusRetriever(
    uri="http://localhost:19530",           # Milvus server URL
    collection_name="documents",            # Collection to search
    embedder=embedder,                      # Embedder for queries
    top_k=5,                                # Number of results
    name="milvus_retriever",
)

print("✅ Milvus Retriever configured")


## 3. Retriever Tool

Use the retriever as a tool for agents:


In [ ]:
from nat.tool.retriever import NatRetrieverTool

# Create a retriever tool for the agent
retriever_tool = NatRetrieverTool(
    retriever=retriever,
    description="Search the document database for relevant information",
    name="search_docs",
)

print("✅ Retriever Tool created")


## 4. RAG Agent Example


In [ ]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

# RAG agent with retriever tool
rag_agent = NatReActAgent(
    tools=[retriever_tool],
    llm=llm,
    verbose=True,
    additional_instructions="Use the search tool to find relevant information before answering.",
)

workflow = NatWorkflow(entrypoint=rag_agent)
print("✅ RAG Workflow created")


In [ ]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "rag_agent.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Saved to: {config_path}")


## Running Milvus

To use Milvus locally:

```bash
# Start Milvus with Docker
docker run -d --name milvus -p 19530:19530 milvusdb/milvus:latest

# Or use Milvus Lite (embedded, no Docker)
# pip install pymilvus
```

## Complete RAG Example

See the full example at `examples/RAG/simple_rag/`:

```bash
nat run --config_file examples/RAG/simple_rag/configs/config.yml \
    --input "What is CUDA?"
```

## Summary

✅ **Embedders** - Convert text to vectors (NIMEmbedder)  
✅ **Retrievers** - Search vector databases (MilvusRetriever)  
✅ **Retriever tools** - Use retrieval in agents  

## Next Steps

- **[08_configuration_guide.ipynb](./08_configuration_guide.ipynb)** - YAML configuration
- **[09_evaluation.ipynb](./09_evaluation.ipynb)** - Evaluate RAG quality
